# 03 — Descriptor-dimension selection

Keep the manually selected layer set and fusion method fixed, reuse its completed 128-D run from Notebook 02, and compare only 64-D, 256-D, and 384-D alternatives.

In [ ]:
from dataclasses import replace
from pathlib import Path
import os

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from matplotlib.ticker import PercentFormatter

from cbir.cache import FeatureShardReader
from cbir.config import FusionConfig, config_to_dict, load_project_config, train_fingerprint
from cbir.data.sfm import Sfm30kMetadata
from cbir.evaluation import evaluate_sfm_verified_pairs, final_cls_descriptors_from_cache
from cbir.experiments import load_results, matching_experiment, read_selection, remove_experiment, save_experiment, write_selection
from cbir.fusion import build_descriptor_head
from cbir.plotting import HorizontalReference, SeriesData, plot_series
from cbir.training import HeadTrainer
from cbir.utils import seed_everything

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'pyproject.toml').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Open this notebook from the project root or notebooks directory.')
os.chdir(PROJECT_ROOT)

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'local.yaml'
cfg = load_project_config(CONFIG_PATH)
CACHE_DIR = cfg.cache.root / 'sfm30k'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / '03_descriptor_dimension_selection'
FIGURES_DIR = OUTPUT_DIR / 'figures'
RESULTS_PATH = OUTPUT_DIR / 'results.json'
LAYER_RESULTS_PATH = PROJECT_ROOT / 'outputs' / '02_layer_set_selection' / 'results.json'
LAYER_SELECTION_PATH = PROJECT_ROOT / 'outputs' / 'selections' / 'layer_set_selection.json'
FINAL_SELECTION_PATH = PROJECT_ROOT / 'outputs' / 'selections' / 'final_model_selection.json'
TRAIN_BATCH_SIZE = cfg.training.batch_size
training_cfg = replace(cfg.training, batch_size=TRAIN_BATCH_SIZE)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

layer_selection = read_selection(LAYER_SELECTION_PATH)
selected_fusion_raw = dict(layer_selection['fusion_config'])
selected_fusion_raw['layer_indices'] = tuple(selected_fusion_raw['layer_indices'])
selected_fusion = FusionConfig(**selected_fusion_raw)
if config_to_dict(training_cfg) != layer_selection['training_config']:
    raise ValueError('Training settings differ from Notebook 02. Rerun layer selection first.')

layer_results = load_results(LAYER_RESULTS_PATH)
selected_record = layer_results['experiments'].get(layer_selection['experiment_name'])
if not isinstance(selected_record, dict):
    raise FileNotFoundError('The selected 128-D result is missing from Notebook 02.')
selected_128 = selected_record['summary']
if selected_128['descriptor_dimension'] != 128:
    raise ValueError('Notebook 02 must select a 128-D experiment before this study.')

reader = FeatureShardReader(CACHE_DIR, preload=True)
if reader.manifest.fingerprint != layer_selection['cache_fingerprint']:
    raise ValueError('The selected layer set does not match this SfM cache.')
metadata = Sfm30kMetadata.from_official_files(
    cfg.sfm.metadata_path,
    cfg.sfm.names_clusters_path,
)
val_ids = metadata.image_ids('val')
val_cases = metadata.build_validation_cases()

print('Selected fusion:', layer_selection['experiment_label'])
print('Training batch size:', TRAIN_BATCH_SIZE)

## Run the remaining dimensions

The 128-D history is inherited from Notebook 02. Matching 64-D, 256-D, and 384-D runs are reused from this notebook’s `results.json`.

In [ ]:
DIMENSIONS_TO_TRAIN = (64, 256, 384)


def result_summary(label, fusion_cfg, history, parameter_count):
    if history.best_epoch is None or history.best_checkpoint is None:
        raise RuntimeError('Experiment did not produce a best checkpoint.')
    metrics = history.epochs[history.best_epoch]
    return {
        'configuration': label,
        'descriptor_dimension': fusion_cfg.output_dim,
        'trainable_parameter_count': parameter_count,
        'descriptor_bytes_fp32': 4 * fusion_cfg.output_dim,
        'descriptor_bytes_fp16': 2 * fusion_cfg.output_dim,
        'best_epoch': history.best_epoch + 1,
        'recall_at_1': metrics['val_recall_at_1'],
        'recall_at_5': metrics['val_recall_at_5'],
        'recall_at_10': metrics['val_recall_at_10'],
        'mrr': metrics['val_mrr'],
    }


def selected_label(output_dim):
    layers = ', '.join(str(index + 1) for index in selected_fusion.layer_indices)
    if selected_fusion.head_kind == 'cls_concat':
        method = 'Multi-level CLS concatenation'
    else:
        method = {
            'uniform': 'Uniform layer weighting',
            'static': 'Static layer weighting',
            'dynamic': 'Dynamic layer weighting',
        }[selected_fusion.gate_mode]
    return f'{method} ({layers}) — {output_dim}-D'


def run_dimension(output_dim):
    name = f'descriptor_{output_dim}'
    fusion_cfg = replace(selected_fusion, output_dim=output_dim)
    run_spec = {
        'cache_fingerprint': reader.manifest.fingerprint,
        'source_experiment': layer_selection['experiment_name'],
        'fusion_config': config_to_dict(fusion_cfg),
        'training_config': config_to_dict(training_cfg),
        'train_fingerprint': train_fingerprint(
            cache_fingerprint=reader.manifest.fingerprint,
            fusion=fusion_cfg,
            training=training_cfg,
        ),
    }
    previous = matching_experiment(RESULTS_PATH, name, run_spec)
    if previous is not None:
        return previous['summary'], previous['history']

    remove_experiment(RESULTS_PATH, name)
    seed_everything(training_cfg.seed)
    head = build_descriptor_head(fusion_cfg)
    parameter_count = sum(parameter.numel() for parameter in head.parameters())
    trainer = HeadTrainer(
        head=head,
        reader=reader,
        train_pairs=metadata.train_pairs,
        fusion_config=fusion_cfg,
        training_config=training_cfg,
        validation_cases=val_cases,
        validation_image_ids=val_ids,
        output_dir=OUTPUT_DIR / 'checkpoints' / name,
    )
    history = trainer.fit()
    summary = result_summary(selected_label(output_dim), fusion_cfg, history, parameter_count)
    save_experiment(
        RESULTS_PATH, name=name, run_spec=run_spec, summary=summary,
        history=history.epochs, checkpoint=history.best_checkpoint,
    )
    return summary, history.epochs


def frozen_baseline():
    run_spec = {'cache_fingerprint': reader.manifest.fingerprint, 'kind': 'frozen_final_cls'}
    previous = matching_experiment(RESULTS_PATH, 'final_cls_384_frozen', run_spec)
    if previous is not None:
        return previous['summary']
    report = evaluate_sfm_verified_pairs(
        final_cls_descriptors_from_cache(reader, val_ids),
        val_ids,
        val_cases,
        query_block_size=cfg.evaluation.query_block_size,
    )
    summary = {
        'configuration': 'Final CLS — 384-D (frozen)',
        'descriptor_dimension': 384,
        'trainable_parameter_count': 0,
        'descriptor_bytes_fp32': 1536,
        'descriptor_bytes_fp16': 768,
        'best_epoch': None,
        'recall_at_1': report.recall_at_1,
        'recall_at_5': report.recall_at_5,
        'recall_at_10': report.recall_at_10,
        'mrr': report.mrr,
    }
    save_experiment(
        RESULTS_PATH, name='final_cls_384_frozen', run_spec=run_spec,
        summary=summary, history=[], checkpoint=None,
    )
    return summary


dimension_results = {
    'final_cls_384_frozen': frozen_baseline(),
    'selected_128': selected_128,
}
dimension_histories = {'selected_128': selected_record['history']}
for output_dim in DIMENSIONS_TO_TRAIN:
    summary, history = run_dimension(output_dim)
    dimension_results[f'descriptor_{output_dim}'] = summary
    dimension_histories[f'descriptor_{output_dim}'] = history

result_table = pd.DataFrame(dimension_results.values())
result_table = result_table.sort_values(['recall_at_1', 'mrr'], ascending=False).reset_index(drop=True)
display(result_table)

In [ ]:
metric_names = ('recall_at_1', 'recall_at_5', 'recall_at_10', 'mrr')
metric_series = {
    item['configuration']: SeriesData(
        x=list(range(len(metric_names))),
        y=[item[name] for name in metric_names],
    )
    for item in dimension_results.values()
}
metric_figure, metric_axis = plot_series(
    metric_series,
    title='Best-checkpoint SfM validation metrics by descriptor dimension',
    xlabel='Validation metric',
    ylabel='Score',
    legend_title='Descriptor configuration',
    fig_size=(12, 6),
    save_path=FIGURES_DIR / 'dimension_selection_metrics.png',
)
metric_axis.set_xticks(range(len(metric_names)), ['R@1', 'R@5', 'R@10', 'MRR'])
metric_axis.yaxis.set_major_formatter(PercentFormatter(1.0))
display(metric_figure)
plt.close(metric_figure)

history_series = {
    dimension_results[name]['configuration']: SeriesData(
        x=[int(epoch['epoch']) + 1 for epoch in history],
        y=[epoch['val_recall_at_1'] for epoch in history],
    )
    for name, history in dimension_histories.items()
}
history_figure, history_axis = plot_series(
    history_series,
    title='SfM validation R@1 during dimension selection',
    xlabel='Epoch',
    ylabel='R@1',
    legend_title='Descriptor configuration',
    horizontal_references={
        'Final CLS — 384-D': HorizontalReference(
            dimension_results['final_cls_384_frozen']['recall_at_1']
        )
    },
    fig_size=(12, 6),
    save_path=FIGURES_DIR / 'dimension_selection_r1_by_epoch.png',
)
history_axis.yaxis.set_major_formatter(PercentFormatter(1.0))
display(history_figure)
plt.close(history_figure)

## Record the final SfM choice

This fixes the model before RevisitOP features are prepared or evaluated.

In [ ]:
FINAL_EXPERIMENT = 'selected_128'

if FINAL_EXPERIMENT not in dimension_results:
    raise KeyError(f'Unknown completed experiment: {FINAL_EXPERIMENT}')
selected = dimension_results[FINAL_EXPERIMENT]
final_fusion = replace(selected_fusion, output_dim=selected['descriptor_dimension'])
source_experiment = (
    layer_selection['experiment_name']
    if FINAL_EXPERIMENT == 'selected_128'
    else f'03_descriptor_dimension_selection/{FINAL_EXPERIMENT}'
)
write_selection(FINAL_SELECTION_PATH, {
    'selected_on': 'Manual student decision after SfM-30k validation inspection only',
    'experiment_name': FINAL_EXPERIMENT,
    'experiment_label': selected['configuration'],
    'cache_fingerprint': reader.manifest.fingerprint,
    'fusion_config': config_to_dict(final_fusion),
    'training_config': config_to_dict(training_cfg),
    'selected_epoch_count': selected['best_epoch'],
    'source_experiment': source_experiment,
})
print('Saved final model selection to', FINAL_SELECTION_PATH)